# 1 assemble the system

In [6]:
import sys
sys.path.append('../utils/')
from utils_models import *

In [7]:
qutip.__version__

'4.7.5'

In [8]:
np.__version__

'1.26.4'

In [9]:
max_ql = 30
max_ol = 75
EJ = 3
EC = EJ/4
EL = EJ/20.5
Er = 8.46111172

g = 0.2
w_d = 8.460155465243822
amp = 0.003

tot_time =660
tlist = np.linspace(0, tot_time, tot_time)[::5]
kappa = 1e-3

system =  FluxoniumOscillatorSystem(
                EJ = EJ,
                EC = EC,
                EL = EL,
                Er = Er,
                g_strength = g, 
                qubit_level = max_ql,
                osc_level = max_ol,
                products_to_keep=[[ql, ol] for ql in range(15) for ol in range(max_ol) ],
                computaional_states = '1,2',
                )

# 2 store the mcsolve jobs

In [12]:
from utils_mcsolve import *

state_leak_dressed = qutip.basis(max_ql * max_ol, system.product_to_dressed[(0,0)])

state_0_dressed = qutip.basis(max_ql * max_ol, system.product_to_dressed[(1,0)])
state_1_dressed = qutip.basis(max_ql * max_ol, system.product_to_dressed[(2,0)])
state_plus_X_dressed = (state_0_dressed +  state_1_dressed).unit()
state_minus_X_dressed = (state_0_dressed - state_1_dressed).unit()
state_plus_Y_dressed = (state_0_dressed + 1j * state_1_dressed).unit()
state_minus_Y_dressed = (state_0_dressed - 1j * state_1_dressed).unit()

initial_states  = [
                    state_leak_dressed,
                   state_0_dressed,
                   state_1_dressed,
                   state_plus_X_dressed,
                   state_minus_X_dressed,
                   state_plus_Y_dressed,
                   state_minus_Y_dressed
                   ]

leakage_products_to_keep = [[ql, ol] for ql in [0,7] for ol in range(max_ol) ]
computational_products_to_keep = [[ql, ol] for ql in [1,2] for ol in range(max_ol) ]
list_of_products_to_keep = [
    leakage_products_to_keep,
    computational_products_to_keep,
    computational_products_to_keep,
    computational_products_to_keep,
    computational_products_to_keep,
    computational_products_to_keep,
    computational_products_to_keep
]


ntraj_per_y0 = 1000
chunk_size = 4
existing_chunk_num = 0
for y0, products_to_keep in zip(initial_states, list_of_products_to_keep):
    system.set_new_product_to_keep(products_to_keep)
    system.set_new_operators_after_setting_new_product_to_keep()
    existing_chunk_num = pack_mcsolve_chunks(
                    y0 = system.truncate_function(y0),
                    tlist = tlist,
                    static_hamiltonian = system.diag_dressed_hamiltonian,
                    drive_terms = [DriveTerm( 
                                driven_op= system.driven_operator,
                                pulse_shape_func=square_pulse_with_rise_fall,
                                pulse_shape_args={
                                    'w_d': w_d ,
                                    'amp': amp,
                                    't_rise': 20,
                                    't_square': tot_time
                                })],                    
                    c_ops = system.c_ops,
                    e_ops = [system.a_trunc , system.a_trunc.dag()*system.a_trunc],
                    ntraj = ntraj_per_y0,
                    existing_chunk_num = existing_chunk_num,
                    chunk_size = chunk_size)
    
pack_pkl_files_to_zip()

# 3 sent to condor and run


# 4 load the mcsolve results, average them

In [13]:
zip_files = [f"result_{i}.zip" for i in range(28)]
n_parts = len(initial_states)
part_length = len(zip_files) // n_parts
zip_file_parts = [zip_files[i * part_length : (i + 1) * part_length] for i in range(n_parts)]
results = []
for part in zip_file_parts:
    results.append(merge_results(part))


progress: 100%|██████████| 4/4 [00:00<00:00, 14.66it/s]


In [ ]:
import pickle

with open('averaged.pkl', 'wb') as f:
    pickle.dump(results,f)

# with open('mcsolve_result_0.03_d0.002_tomo/averaged.pkl', 'rb') as f:
    # results = pickle.load(f)

# 5 Analyzing the eigen states

# 6 Analyzing the tomography states